# Mandatory Assignment 2: Ensemble Models & Artificial Neural Networks
#### Group: 04
Group members: Brage Blomkvist, Patrick Rafael Jemieljanczyk & William Bjørling Nilsen

In [ ]:
# Import required libraries

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy.spatial.distance import cosine
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, TargetEncoder, FunctionTransformer
from sklearn.preprocessing import TargetEncoder
from sklearn.compose import ColumnTransformer
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import TimeSeriesSplit


from sklearn.tree import DecisionTreeClassifier


## Description of Dataset

The dataset used is a set of daily weather observations from Australia. The observations are taken from a span of ten years, between 2007 and 2017. 49 different locations are used to source the data. Most of the locations have a similar number of data entries, but there are some slight differences. From the Kaggle page there seems to be most difference in entries for the periods of 2007-08 and late 2012 to early 2013.

The purpose of the dataset is to see which days it is going to rain on, using the previous day's weather observations. The target value is a binary yes and no, depicting whether it rained or not on a given next day. The target value is stored in the column: "RainTomorrow" Each instance represents a day of weather observations in a time series. Each instance has a value for whether it rained on the day of the observations, and whether it rained on the next day. The threshold for rain that has been used to determine the value of both of these columns is a value of 1mm rain inside the full 24 hours of a day.

The dataset is available on Kaggle through the site publisher Joe Young (https://www.kaggle.com/datasets/jsphyg/weather-dataset-rattle-package/data). The data was retrieved from the Australian Bureau of Meteorology and was collected daily from station observations. The dataset has missing values in multiple features and rows, and even the target value.

The dataset includes multiple guidelines that the developers utilizing the dataset should follow. Since this is a time series, chronological sampling of data should be used. Another guideline is to only fit preprocessing to training data. This will reduce the risk of leakage, since the values, such as mean, will only be derived from the training data itself. In other words, data based on test and validation partitions will not be fitted to the training data. The last guideline is to use more evaluation scores than just accuracy.

### Problem

In [ ]:
# Read the dataset
dataset_AUS = pd.read_csv('Dataset/weatherAUS.csv')
# Uses existing column names baked into csv file.

# Show the first 5 rows to validate that the data import has been successful
dataset_AUS.head()



## Data Processing
Remember the following:
- Remove outliers
- Add correct splitting

In [ ]:
# Overview of dataset
# Look at the implementation details, data types and missing data
dataset_AUS.info()

In [ ]:
# See how many different locations are used.
dataset_AUS.value_counts("Location").nunique()


In [ ]:
# Check if every location has the same amount of observations
dataset_AUS.value_counts("Location")

### Missing Data

In [ ]:
# Look at the amount of missing values
dataset_AUS.isna().sum()

In [ ]:
dataset_AUS.count()

### Outlier removal

In [ ]:
#Outlier removal based on EDA
dataset_AUS = dataset_AUS.loc[(dataset_AUS['Rainfall'] < 350)
                                        & (dataset_AUS['Evaporation'] < 100)
                                        & (dataset_AUS['WindSpeed9am'] < 100)]

### Nested Cross Validation Preparation


In [ ]:
# Creating a temporary dataset
dataset_AUS_temp = dataset_AUS.copy()

# Removing rows where RainTomorrow is missing
dataset_AUS_temp = dataset_AUS_temp.dropna(subset=['RainTomorrow'])

#Converting the Date feature to datetime for sorting
dataset_AUS_temp['Date'] = pd.to_datetime(dataset_AUS_temp['Date'])

#Sorting the dataset chronologically
dataset_AUS_temp = dataset_AUS_temp.sort_values(by=['Date'])
dates = pd.Series(dataset_AUS_temp['Date'])

#Extracting unique dates for Nested Cross Validation
unique_dates = pd.Series(dataset_AUS_temp['Date'].unique())

# Separating into targets and features
targets = dataset_AUS_temp['RainTomorrow']
features = dataset_AUS_temp.drop(columns=['RainTomorrow'])

# Select numerical columns
numerical_features = features.select_dtypes(
    include=['int64', 'float64']
).columns.tolist()

# Select categorical columns
categorical_features = features.select_dtypes(
    include=['object', 'category', 'string']
).columns.tolist()

# Pipelines
to do:
   - Make the processor able to remove certain columns. Can then eventually see which models perform better

   - Can make it an option to delete the feature day_of_year

   - Make an option in the pipeline which can delete the following columns: temp3am, temp9am, year and pressure3pm, day (Need to say what Brage says bout this)


In [ ]:
# Putting the features which need to be encoded into variables as a list. To be used by pipelines to encode
location_features = ['Location']

rain_today_features = ['RainToday']

wind_features = [
    'WindGustDir',
    'WindDir9am',
    'WindDir3pm'
]

date_features = ['Date']

In [ ]:
# Cyclical encoder function
def cyclic_encoder(series, period):
    sin = np.sin(2 * np.pi * series / period)
    cos = np.cos(2 * np.pi * series / period)

    return sin, cos


In [ ]:
wind_map = {
    'N': 1, 'NNE': 2, 'NE': 3, 'ENE': 4,
    'E': 5, 'ESE': 6, 'SE': 7, 'SSE': 8,
    'S': 9, 'SSW': 10, 'SW': 11, 'WSW': 12,
    'W': 13, 'WNW': 14, 'NW': 15, 'NNW': 16
}

WIND_DIRECTIONS = 16
MONTH_PERIOD = 12

In [ ]:
def encode_rain_today(rain_today_dataframe):
    return rain_today_dataframe.replace({'No': 0, 'Yes': 1}).astype(float)

# encode_wind() function needs for-loop because there are three features which need to be encoded by this function.
def encode_wind(wind_data_frame):
    result = pd.DataFrame()

    for column in wind_data_frame.columns:

        wind_numbers = wind_data_frame[column].map(wind_map).astype(float)

        sin, cos = cyclic_encoder(wind_numbers, WIND_DIRECTIONS)
        result[column + '_Sine'], result[column + '_Cosine'] = sin, cos

    return result

# encode_date() function doesn't need a for loop because there is only a single feature with the dates provided.
def encode_date(dataframe):
    result = pd.DataFrame()

    date = pd.to_datetime(dataframe['Date'])
    year = date.dt.year
    month = date.dt.month
    day = date.dt.dayofyear

    is_leap_year = ((year % 4 == 0) & (year % 100 != 0)) | (year % 400 == 0)
    days_in_year = np.select([is_leap_year], [366], default = 365)

    result['Month_Sine'], result['Month_Cosine'] = cyclic_encoder(month, MONTH_PERIOD)

    result['Day_Sine'], result['Day_Cosine'] = cyclic_encoder(day, days_in_year)

    result['Year'] = year

    return result


### Encoding pipelines

In [ ]:
# Numerical pipeline
numerical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

# Location pipeline
location_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', TargetEncoder(target_type='binary',))
])

rain_today_pipeline = Pipeline([
    ('encoder', FunctionTransformer(encode_rain_today)),
    ('imputer', SimpleImputer(strategy='most_frequent'))
])

# Wind pipeline
wind_cyclical_pipeline = Pipeline([
    ('encoder', FunctionTransformer(encode_wind)),
    ('imputer', SimpleImputer(strategy='constant'))
])

# Date pipeline
date_cyclical_pipeline = Pipeline([
    ('encoder', FunctionTransformer(encode_date)),
    ('imputer', SimpleImputer(strategy='median'))
])



### Preprocessor

In [ ]:
# Preprocessor
preprocessor = ColumnTransformer([
    ('numerical', numerical_pipeline, numerical_features),
    ('location', location_pipeline, location_features),
    ('rain_today', rain_today_pipeline, rain_today_features),
    ('wind', wind_cyclical_pipeline, wind_features),
    ('date', date_cyclical_pipeline, date_features)
])

In [ ]:
# Decision tree pipeline
dt_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', DecisionTreeClassifier())
])

# Logistic Regression pipeline
lr_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', LogisticRegression(max_iter=500))
])

# K-nearest neighbours
knn_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', KNeighborsClassifier(
        n_neighbors=5
    ))
])

# train decision tree
dt_pipe.fit(features_train, targets_train)

### Normalization

### Encoding

## Testing Function for Automatic Tuning

In [ ]:
def validation(model):
    train_predictions = model.predict(features_train)
    validation_predictions = model.predict(features_validation)
    print(f'Training Accuracy: {accuracy_score(targets_train["Grades"], train_predictions,):.2f} Validation: {accuracy_score(targets_validation["Grades"], validation_predictions):.2f}')
    print(f'Training Precision: {precision_score(targets_train["Grades"], train_predictions, average="micro"):.2f} Validation: {precision_score(targets_validation["Grades"], validation_predictions, average="micro"):.2f}')
    print(f'Training Recall: {recall_score(targets_train["Grades"], train_predictions, average="micro"):.2f} Validation: {recall_score(targets_validation["Grades"], validation_predictions, average="micro"):.2f}')
    print(f'Training F1: {f1_score(targets_train["Grades"], train_predictions, average="micro"):.2f} Validation: {f1_score(targets_validation["Grades"], validation_predictions, average="micro"):.2f}')

## Nested Cross Validation

In [ ]:
def nested_cross_validation(
        model_pipeline,
        param_grid,
        features,
        targets,
        outer_split=5,
        inner_split=3,
        gap = 1,
        score=recall_score,
        save_path='Models',
):
    #Creating outer and inner CV loop
    outer_cv = TimeSeriesSplit(
        n_splits=outer_split,
        gap=gap
    )
    inner_cv = TimeSeriesSplit(
        n_splits=inner_split,
        gap=gap
        )
    #Storing results
    outer_scores = []
    best_models = []
    best_params = []

    #Outer CV iterates on unique dates so that there won't be any mixing of testing and validation on the same day
    for fold, (trainCvIndices, testIndices) in enumerate(outer_cv.split(unique_dates), 1):

        #Saves the training+validation and test dates
        outer_train_cv_dates = unique_dates.iloc[trainCvIndices]
        outer_test_dates = unique_dates.iloc[testIndices]

        #Checking which rows belong to Training+Validation and Testing
        outer_train_cv_rows = dates.isin(outer_train_cv_dates)
        outer_test_rows = dates.isin(outer_test_dates)

        #Distributing the features and targets for CV and testing
        xTrainCv = features.iloc[outer_train_cv_rows]
        yTrainCv = targets.iloc[outer_train_cv_rows]

        xTest = features.iloc[outer_test_rows]
        yTest = targets.iloc[outer_test_rows]

        #Inner CV dates
        inner_train_cv_rows = dates.iloc[outer_train_cv_rows]

        inner_splits = []
        #Inner CV splits the outer CV training dates into training and validation groups
        for inner_train_index, inner_val_index in inner_cv.split(outer_train_cv_dates):
            inner_train_dates = outer_train_cv_dates[inner_train_index]
            inner_val_dates = outer_train_cv_dates[inner_val_index]

            #Makes lists of indices that are to be included in the training and validation splits for GridSearchCV
            inner_train_rows = np.flatnonzero(
                inner_train_cv_rows.isin(
                    inner_train_dates).to_numpy()
            )
            inner_val_rows = np.flatnonzero(
                inner_train_cv_rows.isin(
                    inner_val_dates).to_numpy()
            )

            inner_splits.append((inner_train_rows, inner_val_rows))

        tuning = GridSearchCV(
            estimator = model_pipeline,
            param_grid = param_grid,
            cv = inner_splits,
            scoring = score,
            n_jobs = -1
        )

        print(
                f"Fold {fold}: "
                f"Training: {outer_train_cv_dates.min()} to {outer_train_cv_dates.max()}, "
                f"Testing: {outer_test_dates.min()} to {outer_test_dates.max()}"
        )

        tuning.fit(xTrainCv, yTrainCv)

        #Finding the best performing models and their parameters
        predictions = tuning.best_estimator_.predict(xTest)
        scores = recall_score(yTest, predictions)

        outer_scores.append(scores)
        best_models.append(tuning.best_estimator_)
        best_params.append(tuning.best_params_)

    return outer_scores, best_models, best_params

In [ ]:
dt_pipe = Pipeline([
    ('preprocessing', preprocessor),
    ('estimator', DecisionTreeClassifier())
])
dt_params = {
    "estimator__max_depth": [12, 15, 20, 25],
    "estimator__max_features": [None, 12, 20, 25],
    "estimator__min_samples_split": [0.005, 0.015, 0.02, 20, 25],
    "estimator__min_samples_leaf": [0.015, 0.025, 0.035, 15, 20]
}
outer_scores, best_models, best_params = nested_cross_validation(dt_pipe, dt_params, features, targets, outer_split=5, inner_split=3)

In [ ]:
print(outer_scores)

## Baseline Models

In [ ]:
# (Automated tuning)

## Advanced Models

## Advanced Neural Networks

## Evaluation & Comparison

### Testing

#### Combined Results Table

### Confusion Matrices & Additional Plots

### Conclusion